# Experiment 2 analysis

Reads the consolidated CSVs under `data/outputs/Experiment 2/_analysis/` and renders, in order:

1. Runs per model.
2. Per-defect-type ? model recall from `defect_recall.csv`.
3. One detailed table per model.
4. Heatmap of per-defect-type recall.
5. Offline test-set score scatter from `test_scores.csv`.

The notebook does not read raw experiment output folders or the source dataset tree.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks and aux scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hardware_equivalence import normalize_latency

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams["figure.dpi"] = 110

ALL_MODELS = [
    "patchcore", "padim", "subspacead", "stfpm",
    "csflow", "draem", "rd4ad",
]

EXP_ROOT = PROJECT_ROOT / "data" / "outputs" / "Experiment 2"
ANALYSIS_DIR = EXP_ROOT / "_analysis"
RUNS_CSV = ANALYSIS_DIR / "runs_cache.csv"
DEFECT_RECALL_CSV = ANALYSIS_DIR / "defect_recall.csv"
SCORES_CSV = ANALYSIS_DIR / "test_scores.csv"

print(f"Runs CSV: {RUNS_CSV}")
print(f"Defect recall CSV: {DEFECT_RECALL_CSV}")
print(f"Scores CSV: {SCORES_CSV}")


In [ ]:
df = pd.read_csv(RUNS_CSV)
defect_df = pd.read_csv(DEFECT_RECALL_CSV)
score_df = pd.read_csv(SCORES_CSV, low_memory=False)
normalize_latency(df)
MODELS_PRESENT = sorted(df["model"].dropna().unique()) if not df.empty else []
print(f"Loaded {len(df)} consolidated run rows across models: {MODELS_PRESENT}")
print(f"Loaded {len(defect_df)} defect-recall rows and {len(score_df)} score rows.")


## 1. Runs per model

Count of model evaluations found under `Experiment 2/jobB_val_defect_V1/`. The Polymer-sheet dataset is a single product, so there is no per-category breakdown.

In [ ]:
runs_per_model = (
    df.groupby("model").size().reindex(ALL_MODELS, fill_value=0).to_frame("runs")
)
runs_per_model.index.name = "model"
runs_per_model

## 2. Per-defect-type recall — defect × model

Recall broken down by ground-truth defect class. Defect types replace the per-category dimension from Experiment 1; the support column on the right shows the number of NG samples of each type present in the test split.

In [ ]:
defect_types = sorted(defect_df["defect_type"].dropna().unique())

recall_matrix = defect_df.pivot_table(
    index="defect_type", columns="model", values="recall", aggfunc="mean"
).reindex(index=defect_types, columns=MODELS_PRESENT)

support_series = defect_df.groupby("defect_type")["support"].max().reindex(defect_types).fillna(0).astype(int)
recall_matrix_display = recall_matrix.copy()
recall_matrix_display["support"] = support_series
recall_matrix_display.index.name = "defect_type"
recall_matrix_display


## 3. Per-model detailed table

One table per model. With a single product, each model produces a single row; columns are the headline run-level metrics. The `train_samples` / `val_samples` / `test_samples` triple is the offline split: the threshold is fit on the validation split and metrics are reported on the held-out test split.

In [ ]:
PER_MODEL_COLS = [
    "threshold_value", "threshold_mode",
    "train_samples", "val_samples", "test_samples",
    "auroc", "aupr", "precision", "recall", "f1", "accuracy",
    "mean_score_ok", "mean_score_ng",
    "mean_latency_ms", "throughput_fps",
]

for model in MODELS_PRESENT:
    sub = (
        df[df["model"] == model]
        .set_index("experiment")[PER_MODEL_COLS]
        .sort_index()
    )
    print(f"\n=== {model} ({len(sub)} runs) ===")
    display(sub)

## 4. Heatmap — per-defect-type recall (defect × model)

In [ ]:
def defect_recall_heatmap(title: str, *, vmin: float = 0.0, vmax: float = 1.0) -> None:
    mat = recall_matrix.reindex(index=defect_types, columns=MODELS_PRESENT)
    fig, ax = plt.subplots(
        figsize=(1.1 * len(MODELS_PRESENT) + 2, 0.55 * len(defect_types) + 1.5)
    )
    masked = np.ma.masked_invalid(mat.values.astype(float))
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="#e5e5e5")
    im = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(MODELS_PRESENT)))
    ax.set_xticklabels(MODELS_PRESENT, rotation=45, ha="right")
    ax.set_yticks(range(len(defect_types)))
    ax.set_yticklabels(defect_types)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if pd.notna(v):
                ax.text(
                    j, i, f"{v:.2f}",
                    ha="center", va="center", fontsize=8,
                    color="white" if v < (vmin + vmax) / 2 else "black",
                )
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    plt.show()


defect_recall_heatmap("Recall \u2014 defect type \u00d7 model")

## 5. Offline test-set scores — OK vs NG per model

Scatter of the per-image anomaly score over the held-out **test split** of each run (no warm-up / no calibration phase in offline evaluation). Green = OK ground truth, red = NG. The dashed line marks the threshold fit on the validation split (`val_f1` by default).

In [ ]:
SCATTER_COLS = 3
n = len(MODELS_PRESENT)
nrows = math.ceil(n / SCATTER_COLS)
fig, axes = plt.subplots(
    nrows, SCATTER_COLS,
    figsize=(SCATTER_COLS * 4.4, nrows * 3.0),
    squeeze=False,
)
fig.suptitle("Offline test-set scores (OK vs NG) per model", fontsize=12)

for i, model in enumerate(MODELS_PRESENT):
    ax = axes[i // SCATTER_COLS][i % SCATTER_COLS]
    row = df[df["model"] == model].iloc[0]
    recs = score_df[
        (score_df["experiment"] == row["experiment"])
        & (score_df["model"] == row["model"])
    ].sort_values("sample_idx")
    if recs.empty:
        ax.set_visible(False)
        continue
    scores = recs["score"].to_numpy(dtype=float)
    labels = recs["label"].to_numpy()
    idx = recs["sample_idx"].to_numpy(dtype=int)
    ok = labels == 0
    ng = labels == 1
    ax.scatter(idx[ok], scores[ok], s=8, c="tab:green", alpha=0.55, label="OK")
    ax.scatter(idx[ng], scores[ng], s=8, c="tab:red", alpha=0.55, label="NG")
    thr = row["threshold_value"]
    if pd.notna(thr):
        ax.axhline(thr, color="black", lw=0.7, ls="--", label=f"thr={thr:.2f}")
    ax.set_title(model, fontsize=10)
    ax.set_xlabel("test sample index", fontsize=8)
    ax.set_ylabel("score", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc="best")
    ax.grid(alpha=0.25)

for j in range(n, nrows * SCATTER_COLS):
    axes[j // SCATTER_COLS][j % SCATTER_COLS].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
